# 📖 Notebook 4 — Real-World CDN Pitfalls & Patterns

Notebooks 1–3 covered the happy path: origin → edge → user, with `Cache-Control`, ETags, and invalidation. This notebook covers the things that bite real teams in production:

1. **Cache stampede** (a.k.a. *thundering herd* / *dogpile*) — and how `proxy_cache_lock` saves your origin.
2. **Stale-while-revalidate** — keeping users fast even when the origin is slow or down.
3. **The `Vary` header trap** — accidentally caching one user's response for everyone.
4. **`Cache-Control` directives cheat-sheet** — `private`, `no-cache`, `no-store`, `s-maxage`, `must-revalidate`.
5. **Never cache authenticated responses** — the classic CDN security bug.

> 🛡️ Analogy: notebooks 1–3 taught you to *drive*. This notebook teaches you the *bad weather*: skids, blind spots, and the one button you should never press.

## 🛠️ Setup

Same as the previous notebooks — start the lab from `01-foundations/cdn/`:

```bash
docker compose up -d --build
uv sync
```

Then select the `.venv` kernel (top-right of this notebook). Reload the VS Code window if it doesn't appear.

In [ ]:
import time
import threading
import subprocess
import httpx

ORIGIN = "http://localhost:8000"
EDGE1  = "http://localhost:8081"

def timed_get(url: str, **kwargs):
    t0 = time.perf_counter()
    r = httpx.get(url, timeout=30.0, **kwargs)
    return (time.perf_counter() - t0) * 1000, r

def purge_edge(container: str = "cdn-edge1") -> None:
    """Wipe the on-disk cache of an edge container."""
    subprocess.run(
        ["docker", "exec", container, "sh", "-c", "rm -rf /var/cache/nginx/edge/*"],
        check=True, capture_output=True,
    )

## 1️⃣ Cache stampede: the thundering herd

Imagine a viral tweet hits at 3 pm. 10,000 users ask for the same image at the same instant. The edge cache is **cold** for that asset. What happens?

- ❌ **Naive cache**: *every one* of those 10,000 misses forwards to the origin simultaneously. The origin melts. This is the **cache stampede**.
- ✅ **Smart cache**: only the *first* miss is forwarded; the other 9,999 wait for that single response and reuse it.

Our nginx config already does the right thing thanks to one line:

```nginx
proxy_cache_lock on;
```

Let's prove it. We'll fire 20 simultaneous requests at a fresh cache key and count how many actually reached the origin.

In [ ]:
key = f"?stampede={int(time.time())}"
url = f"{EDGE1}/assets/hello.txt{key}"

results = []
lock = threading.Lock()

def worker(i):
    ms, r = timed_get(url)
    with lock:
        results.append((i, ms, r.headers.get("x-cache-status")))

# Launch 20 threads at (almost) the same instant.
barrier = threading.Barrier(20)
def gated(i):
    barrier.wait()
    worker(i)

threads = [threading.Thread(target=gated, args=(i,)) for i in range(20)]
for t in threads: t.start()
for t in threads: t.join()

from collections import Counter
statuses = Counter(s for _, _, s in results)
print(f"cache statuses across 20 concurrent requests: {dict(statuses)}")
print(f"slowest request: {max(ms for _, ms, _ in results):.0f} ms")
print(f"fastest request: {min(ms for _, ms, _ in results):.0f} ms")

### ☝️ What just happened?

You should see only **one** `MISS` (the lucky thread that actually went to the origin) and the rest as `HIT` — because nginx made them wait the ~500 ms for that single upstream call to finish, then served all of them from the freshly-warmed cache.

Without `proxy_cache_lock`, you'd see 20 MISSes and the origin would have done 20× the work. At Twitter scale that's the difference between *one* origin call per viral asset and *millions*.

> 💡 In application caches (Redis, Memcached) the same problem appears as the **dogpile effect**, and is fixed with locks, request coalescing, or **probabilistic early expiration** (refresh slightly before TTL with some probability).

## 2️⃣ Stale-while-revalidate: serve fast, refresh in the background

What if the origin is slow or briefly down? You don't want every user to wait — or worse, see an error. Two related ideas help:

- **`stale-while-revalidate`** (HTTP standard, RFC 5861): *"You may serve a stale copy for up to N seconds while you refresh in the background."*
- **`proxy_cache_use_stale`** (nginx): serve a stale entry if the origin returns an error or times out.

Our nginx config already enables the second one:

```nginx
proxy_cache_use_stale error timeout updating http_500 http_502 http_503 http_504;
```

Let's prove it. We'll prime the edge cache, then **stop the origin entirely**, then ask the edge again — it should still serve the stale copy instead of returning 502.

In [ ]:
key = f"?swr={int(time.time())}"
url = f"{EDGE1}/assets/hello.txt{key}"

ms, r = timed_get(url)
print(f"prime cache: {ms:.0f} ms  status={r.status_code}  cache={r.headers.get('x-cache-status')}")

print("\nstopping the origin container...")
subprocess.run(["docker", "stop", "cdn-origin"], check=True, capture_output=True)

try:
    # Wait long enough that the entry is past max-age=30 → would normally need revalidation.
    print("sleeping 35s so the cache entry goes past its max-age...")
    time.sleep(35)
    ms, r = timed_get(url)
    print(f"\nedge while origin is DOWN: status={r.status_code}  {ms:.0f} ms  cache={r.headers.get('x-cache-status')}")
    print(f"body bytes={len(r.content)}  (edge served the stale copy instead of erroring out)")
finally:
    print("\nrestarting origin...")
    subprocess.run(["docker", "start", "cdn-origin"], check=True, capture_output=True)
    # Wait for healthcheck.
    for _ in range(20):
        try:
            httpx.get(f"{ORIGIN}/health", timeout=2.0).raise_for_status()
            print("origin is back")
            break
        except Exception:
            time.sleep(1)

### ☝️ What just happened?

Even though the origin was completely offline and the cache entry had passed its `max-age`, the edge returned the **last good copy** instead of a 5xx error. Your users see a slightly-stale page; you see no incident.

This is one of the most loved features of CDNs in production. Cloudflare's *Always Online*, Fastly's *serve stale on error*, and AWS CloudFront's *origin failover* all build on this idea.

> ⚠️ Caveat: stale-while-revalidate is great for *read-mostly* assets (HTML, images, API responses you can tolerate being a few seconds old). It is **not** appropriate for content that must be exact (account balances, order status).

## 3️⃣ The `Vary` header trap

Caches use the URL as the cache key. If your response actually depends on a **request header** (e.g. `Accept-Encoding`, `Accept-Language`, or — worst case — `Cookie`), the cache will happily serve the wrong version to the wrong user *unless* you tell it to vary on that header.

The cure is the `Vary` header:

```
Vary: Accept-Encoding
Vary: Accept-Language, Accept-Encoding
```

It tells caches: *"the cache key is (URL, value-of-this-header)"*. The classic disasters:

| Symptom                                                      | Likely cause                                      |
|--------------------------------------------------------------|---------------------------------------------------|
| French users see English pages                               | Server returns translated content but no `Vary: Accept-Language` |
| Some users get gzipped garbage                               | Mixed gzipped/identity responses without `Vary: Accept-Encoding` |
| **One user's logged-in page leaks to everyone**              | Per-cookie response without `Vary: Cookie` (or, better, `Cache-Control: private`) |

> 🛑 **Rule of thumb**: never cache anything that depends on `Cookie` at a shared cache. `Vary: Cookie` technically works, but it explodes your cache (one entry per cookie value). Prefer `Cache-Control: private`.

## 4️⃣ `Cache-Control` directive cheat-sheet

The directives in `Cache-Control` look harmless but mean very different things.

| Directive             | Plain-English meaning                                                  | Who it's for       |
|-----------------------|------------------------------------------------------------------------|--------------------|
| `public`              | Any cache may store this — browser, CDN, anyone.                       | Browsers + CDN     |
| `private`             | Only the end-user's browser may cache this. CDNs/proxies must skip it. | Browsers only      |
| `no-store`            | Don't write this anywhere. Ever. (For sensitive data.)                 | Everyone (a ban)   |
| `no-cache`            | You may store it, but **revalidate every time** before serving it.     | Browsers + CDN     |
| `max-age=N`           | Treat as fresh for N seconds (browser + CDN).                          | Browsers + CDN     |
| `s-maxage=N`          | Same, but **only for shared caches** (CDN). Overrides `max-age`.       | CDN only           |
| `must-revalidate`     | Once stale, you *must* check origin — never serve stale on error.      | Everyone           |
| `stale-while-revalidate=N` | OK to serve stale up to N seconds while refreshing in background. | Everyone           |
| `immutable`           | Promise: this URL's content will *never* change. Don't even revalidate.| Browsers           |

Two patterns cover ~90% of real apps:

```http
# Hash-fingerprinted static asset, e.g. /static/app.8f3c1.js
Cache-Control: public, max-age=31536000, immutable

# HTML page (the entry point that *references* the hashed assets)
Cache-Control: public, max-age=0, s-maxage=60, stale-while-revalidate=300
```

The HTML changes often, so you cache it short at the CDN (`s-maxage=60`) and effectively bypass the browser cache (`max-age=0`). The hashed JS/CSS files never change for a given URL, so you cache them *forever*.

## 5️⃣ The classic security bug: caching authenticated responses

This is the #1 way CDNs cause data breaches. The setup:

1. Logged-in user **Alice** requests `/account` from the CDN. The origin returns Alice's account page.
2. The origin forgets to set `Cache-Control: private` (or sets `public`). The CDN caches the response keyed by URL.
3. Logged-in user **Bob** requests `/account`. Same URL → **HIT** → Bob sees Alice's account.

It has happened to Steam, Target, multiple banks. Defenses, in order:

1. **Always set `Cache-Control: private` (or `no-store`) on per-user responses.** Belt-and-braces: also set `Vary: Cookie`.
2. **Don't proxy authenticated routes through the CDN at all.** Many setups serve `/static/*` through the CDN and `/api/*` straight to the origin.
3. **Strip cookies from the cache key for static assets**, and **bypass the cache entirely when the request has an auth cookie** for dynamic routes. nginx:

```nginx
proxy_cache_bypass $cookie_sessionid;   # don't serve from cache if sessionid is set
proxy_no_cache     $cookie_sessionid;   # don't store responses for those requests
```

> 🚨 If you remember nothing else from this lab: **shared caches + cookies = sharp knife**. Treat any response that depends on a logged-in user as `private` by default.

## 📦 Recap

| Pitfall                       | Defense                                                       |
|-------------------------------|---------------------------------------------------------------|
| Cache stampede on cold key    | `proxy_cache_lock on;` (nginx) / request coalescing in code   |
| Origin briefly down           | `proxy_cache_use_stale ...;` / `stale-while-revalidate`       |
| Wrong language/encoding served | `Vary: Accept-Language, Accept-Encoding`                     |
| Per-user data leaks across users | `Cache-Control: private`, `proxy_cache_bypass $cookie_*`   |
| Static asset stale forever    | Hash-fingerprinted URLs + `immutable`                         |

You've now seen the same patterns Cloudflare, Fastly, CloudFront, Akamai, and Bunny.net build their products around — at toy scale, but with the same headers and behaviours.

🎉 That's the end of the CDN lab. Suggested next steps:

- Try changing `proxy_cache_lock` to `off` in `edge/nginx.conf`, restart the edges, and rerun the stampede experiment in §1 — you'll see lots of MISSes.
- Try setting `Cache-Control: private` on the origin and watch nginx refuse to cache it (`X-Cache-Status: BYPASS`).
- Compare to a real CDN: deploy a static site behind Cloudflare or CloudFront and inspect `cf-cache-status` / `x-cache` headers in your browser devtools.